# Fine-tune YOLO11 on MIO-TCD
Fine-tunes a YOLO11 checkpoint using **train** and **val** only. The frozen test split is deliberately not loaded, used for early stopping, or evaluated here. Output: Ultralytics run (including `weights/best.pt` and `weights/last.pt`) under `outputs/mio_tcd/train/`.

## Config

In [ ]:
MODEL_NAME = 'yolo11n.pt'  # Change to yolo11s.pt if desired
IMG_SIZE = 640
EPOCHS = 50
BATCH_SIZE = 16
DEVICE = None
WORKERS = 4
SEED = 42
PROJECT_DIR = 'outputs/mio_tcd/train'
RUN_NAME = 'yolo11n_mio_v1'
PATIENCE = 15
FORCE_OVERWRITE = False

## Imports and validation

In [ ]:
from pathlib import Path
import random, sys, numpy as np, torch, ultralytics, yaml
from ultralytics import YOLO
sys.path.insert(0, str(Path.cwd()))
from mio_tcd_utils import PROJECT_ROOT
DATA_YAML = PROJECT_ROOT / 'data/mio_tcd/yolo/mio_tcd.yaml'
TRAIN_LIST = PROJECT_ROOT / 'data/mio_tcd/splits/train.txt'; VAL_LIST = PROJECT_ROOT / 'data/mio_tcd/splits/val.txt'
if not DATA_YAML.is_file() or not TRAIN_LIST.is_file() or not VAL_LIST.is_file(): raise FileNotFoundError('Run 01–03 notebooks first.')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Torch:', torch.__version__, '| Ultralytics:', ultralytics.__version__, '| CUDA:', torch.cuda.is_available())
print('Only train/val are passed to training; frozen test is excluded.')

## Train

In [ ]:
run_dir = PROJECT_ROOT / PROJECT_DIR
if (run_dir / RUN_NAME).exists() and not FORCE_OVERWRITE: raise FileExistsError('Run directory exists; choose a new RUN_NAME or set FORCE_OVERWRITE=True.')
model = YOLO(MODEL_NAME)
train_results = model.train(data=str(DATA_YAML), imgsz=IMG_SIZE, epochs=EPOCHS, batch=BATCH_SIZE, device=DEVICE, workers=WORKERS, seed=SEED, patience=PATIENCE, project=str(run_dir), name=RUN_NAME, exist_ok=FORCE_OVERWRITE, plots=True, val=True)
print('Training completed:', run_dir / RUN_NAME)
print('Expected checkpoints:', run_dir / RUN_NAME / 'weights/best.pt', 'and last.pt')

## Summary

In [ ]:
results_csv = run_dir / RUN_NAME / 'results.csv'
if results_csv.is_file():
    import pandas as pd
    history = pd.read_csv(results_csv); display(history.tail()); history.plot(x='epoch', y=[c for c in history.columns if 'loss' in c or 'metrics/mAP50' in c], figsize=(12, 5))
else: print('Ultralytics results.csv was not found; inspect the run directory.')